# Breakout Trend 01 — hourly breakout strategy on Deribit perpetuals and long options

Pre-registration: `discussions/Breakout_Trend/001_claude_branch_preregistration.md`. Strategy, costs, windows and the
decision rule are fixed in `src/data_engineering/modeling/breakout_trend.py` (`registration()` below).

- **Perpetual leg:** 1 unit of underlying per trade, taker fee + 1 bp slippage per side, funding ignored.
- **Options leg:** long ATM call (long signal) / put (short signal), first daily expiry ≥ 24 h away, delta-matched size,
  model-priced over the whole history (IV $=k\cdot\hat\sigma_{\text{EWMA}}$, half-spread $h$ by TTE, both calibrated on the
  collector's real quotes); real-quote check on the collector window.
- **Windows:** development 2024-09-01 → 2025-12-31, test 2026-01-01 → end of data.
- **Equity:** sum of returns on 1 unit of entry notional, in % (not compounded).

Environment: `BT01_ASSETS` (comma list), `BT01_N_NULL` (default 200), `BT01_SYNTHETIC=1` (synthetic smoke run, outputs to a temp dir).

In [ ]:
from pathlib import Path
import json, os, sys, tempfile, time, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from dotenv import load_dotenv

warnings.filterwarnings('ignore', category=RuntimeWarning)
pd.set_option('display.max_columns', 40); pd.set_option('display.width', 250)
load_dotenv()
PROJECT_ROOT = Path.cwd().resolve()
while PROJECT_ROOT != PROJECT_ROOT.parent and not (PROJECT_ROOT / 'src' / 'data_engineering').exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
sys.path.insert(0, str(PROJECT_ROOT / 'src'))
from data_engineering.modeling import breakout_trend as bt

SYNTHETIC = os.getenv('BT01_SYNTHETIC', '').strip() == '1'
ASSETS = [a.strip().upper() for a in os.getenv('BT01_ASSETS', 'BTC,ETH,SOL,XRP,AVAX,TRX,HYPE').split(',') if a.strip()]
N_NULL = int(os.getenv('BT01_N_NULL', str(bt.N_NULL)))
P = bt.Params()
if SYNTHETIC:
    DATA_ROOT = None
    OUT = Path(tempfile.mkdtemp(prefix='bt01_'))
else:
    DATA_ROOT = Path(os.environ['DERIBIT_DATA_ROOT']).expanduser().resolve()
    OUT = PROJECT_ROOT / 'notebooks' / 'Breakout_Trend' / '01'
OUT.mkdir(parents=True, exist_ok=True)
print('BT 01', bt.VERSION, '| synthetic' if SYNTHETIC else '', '| out', OUT)
print(bt.selftest())
print(json.dumps(bt.registration(P), indent=2))

## 1. Data: 5-minute OHLC history and collector option quotes

In [ ]:
frames, quotes, coverage = {}, {}, []
for i, a in enumerate(ASSETS):
    if SYNTHETIC:
        df = bt.synthetic_5m(n_days=770, seed=100 + i, start='2024-09-01')
        q = None
    else:
        path = DATA_ROOT / 'history' / 'ohlc_5m' / f'{a}.csv.gz'
        if not path.exists():
            print(f'{a}: no OHLC history at {path} -> skipped'); continue
        df = pd.read_csv(path)
        q = None
    fr = bt.decision_frame(df, P)
    fr = fr[fr['state_time'] >= bt.DEV_START - pd.Timedelta(days=6)].reset_index(drop=True)   # warm-up kept
    frames[a] = fr
    if SYNTHETIC:
        qa = bt.synthetic_quotes(fr, k=1.1 + 0.05 * i, rel_half=0.03 + 0.01 * (i % 3),
                                 start=str((fr['state_time'].iloc[-1] - pd.Timedelta(days=21)).date()))
        qinfo = {'status': 'synthetic', 'rows': len(qa)}
    else:
        qa, qinfo = bt.load_option_quotes(DATA_ROOT, a)
    quotes[a] = qa
    coverage.append({'asset': a, 'first_bar': fr['state_time'].min(), 'last_bar': fr['state_time'].max(),
                     'bars': len(fr), 'valid_share': round(float(fr['valid'].mean()), 4),
                     'source': df['source'].iloc[0] if 'source' in df else 'synthetic',
                     'quotes': qinfo.get('rows', 0), 'quote_family': qinfo.get('family'),
                     'quotes_first': qinfo.get('first'), 'quotes_last': qinfo.get('last'),
                     'quote_grids': qinfo.get('by_grid')})
if not frames:
    raise RuntimeError('No 5-minute OHLC history found under DERIBIT_DATA_ROOT/history/ohlc_5m. '
                       'Run first:  python tools\\download_ohlc_history.py --start 2024-08-25')
cov = pd.DataFrame(coverage); display(cov)
END = max(fr['state_time'].max() for fr in frames.values()) + pd.Timedelta(minutes=5)
WINDOWS = {'dev': (bt.DEV_START, bt.TEST_START), 'test': (bt.TEST_START, END), 'full': (bt.DEV_START, END)}

## 2. Strategy, perpetual leg, option calibration and model-priced option leg

In [ ]:
res, calib = {}, []
for a, fr in frames.items():
    t0 = time.time()
    tr = bt.perp_returns(bt.run_strategy(fr, P))
    vol = bt.ewma_vol(fr)
    eq_perp = bt.perp_equity(fr, tr)
    q = quotes[a]
    model, info = (bt.calibrate_option_model(q, fr, vol) if len(q) else (None, {'k': np.nan}))
    ok_opt = model is not None and np.isfinite(model.k) and all(np.isfinite(model.h))
    if ok_opt:
        pcs, eq_opt = bt.model_option_leg(fr, tr, model, vol)
        real = bt.real_option_leg(fr, tr, q)
    else:
        pcs, eq_opt, real = pd.DataFrame(), np.full(len(fr), np.nan), pd.DataFrame()
    res[a] = dict(tr=tr, eq_perp=eq_perp, eq_opt=eq_opt, pcs=pcs, real=real, model=model, ok_opt=ok_opt)
    calib.append({'asset': a, **{k: v for k, v in info.items() if k != 'h'},
                  'h_0_6h': info.get('h', [np.nan] * 3)[0], 'h_6_24h': info.get('h', [np.nan] * 3)[1],
                  'h_24_48h': info.get('h', [np.nan] * 3)[2], 'options_leg': ok_opt})
    print(f'{a}: {len(tr)} trades, options {"ON" if ok_opt else "OFF"}, {time.time() - t0:.1f}s')
calib = pd.DataFrame(calib); display(calib)

## 3. Metrics by window

In [ ]:
rows = []
daily = {}
for a, r in res.items():
    fr, tr = frames[a], r['tr']
    ed_p = bt.daily_equity(fr, r['eq_perp'])
    ed_o = bt.daily_equity(fr, r['eq_opt'])
    bh = bt.buy_hold_daily(fr)
    daily[a] = pd.DataFrame({'perp': ed_p, 'options': ed_o, 'buy_hold': bh})
    for w, (s, e) in WINDOWS.items():
        bhm = bt.period_metrics(bh, s, e)
        base = {'asset': a, 'window': w, 'time_in_market': bt.time_in_market(fr, tr, s, e),
                'buy_hold_sharpe': bhm['sharpe'], 'buy_hold_ret': bhm['pnl']}
        rows.append({**base, 'leg': 'perp', **bt.period_metrics(ed_p, s, e), **bt.trade_metrics(tr, 'perp_net', s, e)})
        if r['ok_opt']:
            pc = r['pcs'].merge(tr[['entry_time', 'exit_time']].reset_index().rename(columns={'index': 'trade_id'}), on='trade_id')
            tro = pc.groupby('trade_id').agg(opt_ret=('opt_ret', 'sum'), dir=('dir', 'first'),
                                             entry_time=('entry_time', 'first'), exit_time=('exit_time', 'first'))
            rows.append({**base, 'leg': 'options_model', **bt.period_metrics(ed_o, s, e), **bt.trade_metrics(tro, 'opt_ret', s, e)})
metrics = pd.DataFrame(rows)
show = ['asset', 'window', 'leg', 'pnl', 'sharpe', 'max_dd', 'n_trades', 'trades_per_year', 'win_rate', 'avg_win',
        'avg_loss', 'pnl_long', 'pnl_short', 'time_in_market', 'buy_hold_sharpe', 'buy_hold_ret']
fmt = metrics[show].copy()
for c in ('pnl', 'max_dd', 'avg_win', 'avg_loss', 'pnl_long', 'pnl_short', 'buy_hold_ret'):
    fmt[c] = 100 * fmt[c]          # % of notional
display(fmt.round(3))

## 4. Coin-flip null (perpetual leg): random entries from flat, same exits

In [ ]:
nulls = {}
null_daily = {}
t0 = time.time()
for a, fr in frames.items():
    rate = bt.flat_entry_rate(fr, res[a]['tr'])
    ss = np.random.SeedSequence([bt.NULL_SEED, ASSETS.index(a)])
    eqs, rows = [], []
    for run, child in enumerate(ss.spawn(N_NULL)):
        t = bt.run_strategy(fr, P, mode='random', p_entry=rate, rng=np.random.default_rng(child))
        ed = bt.daily_equity(fr, bt.perp_equity(fr, t))
        eqs.append(ed)
        for w, (s, e) in WINDOWS.items():
            m = bt.period_metrics(ed, s, e)
            rows.append({'run': run, 'window': w, 'pnl': m['pnl'], 'sharpe': m['sharpe']})
    nulls[a] = pd.DataFrame(rows)
    null_daily[a] = pd.concat(eqs, axis=1, keys=range(N_NULL))
    print(f'{a}: p_entry {rate:.5f}, {N_NULL} runs, {time.time() - t0:.0f}s elapsed')

# pooled equal-weight portfolio (sum of per-asset equity), null runs paired by run index
idx = pd.concat(daily.values(), axis=1).index
pool = pd.DataFrame({leg: sum(daily[a][leg].reindex(idx).ffill().fillna(0.0) for a in daily if daily[a][leg].notna().any())
                     for leg in ('perp', 'options')})
pool_null = sum(null_daily[a].reindex(idx).ffill().fillna(0.0) for a in null_daily)
nulls['POOLED'] = pd.DataFrame([{'run': r, 'window': w, **{k: v for k, v in bt.period_metrics(pool_null[r], s, e).items() if k in ('pnl', 'sharpe')}}
                                for r in range(N_NULL) for w, (s, e) in WINDOWS.items()])

drows = []
for a in list(frames) + ['POOLED']:
    ed = pool['perp'] if a == 'POOLED' else daily[a]['perp']
    for w, (s, e) in WINDOWS.items():
        m = bt.period_metrics(ed, s, e)
        x = nulls[a][nulls[a]['window'] == w]
        pn = bt.null_share(nulls[a], w, m['sharpe'])
        drows.append({'asset': a, 'window': w, 'pnl_pct': 100 * m['pnl'], 'sharpe': m['sharpe'],
                      'null_sharpe_median': float(x['sharpe'].median()), 'null_sharpe_q95': float(x['sharpe'].quantile(0.95)),
                      'null_share_ge': pn, 'null_pnl_median_pct': 100 * float(x['pnl'].median()),
                      'verdict': bt.verdict(m['pnl'], pn)})
decision_tab = pd.DataFrame(drows)
display(decision_tab.round(3))
primary = decision_tab[(decision_tab.asset == 'POOLED') & (decision_tab.window == 'test')].iloc[0].to_dict()
print('PRIMARY (pooled, test window):', primary['verdict'])

## 5. Equity curves: perpetual vs options, per asset

In [ ]:
panels = list(daily) + ['POOLED']
ncol = 2
nrow = int(np.ceil(len(panels) / ncol))
fig, axes = plt.subplots(nrow, ncol, figsize=(14, 3.2 * nrow), sharex=True)
axes = np.atleast_1d(axes).ravel()
for ax, a in zip(axes, panels):
    d = pool if a == 'POOLED' else daily[a]
    d = d[d.index >= bt.DEV_START]
    ax.plot(d.index, 100 * d['perp'], color='#1f5aa6', lw=1.3, label='Perpetual')
    if d['options'].notna().any():
        ax.plot(d.index, 100 * d['options'], color='#c2571a', lw=1.3, label='Options (model-priced)')
    ax.axvline(bt.TEST_START, color='k', lw=0.8, ls='--')
    if a != 'POOLED' and len(quotes.get(a, [])):
        ax.axvspan(quotes[a]['state_time'].min(), quotes[a]['state_time'].max(), color='0.85', zorder=0)
    ax.axhline(0, color='0.5', lw=0.6)
    ax.set_title(a if a != 'POOLED' else 'Pooled (sum of assets)', fontsize=10, loc='left')
    ax.set_ylabel('% of notional')
    ax.grid(alpha=0.25)
for ax in axes[len(panels):]:
    ax.set_visible(False)
axes[0].legend(loc='upper left', fontsize=8, frameon=False)
fig.suptitle('Breakout strategy equity, net of costs (dashed: test start 2026-01-01; grey: collector option quotes)', fontsize=11)
plt.tight_layout(); plt.show()

## 6. Collector window: model-priced vs real-quote option trades

In [ ]:
crows = []
fig, ax = plt.subplots(figsize=(7, 4))
for a, r in res.items():
    if not r['ok_opt'] or r['real'].empty:
        continue
    c = r['real'].merge(r['pcs'][['trade_id', 'piece', 'opt_ret']], on=['trade_id', 'piece'], how='left')
    c['asset'] = a
    crows.append(c)
    ok = c.dropna(subset=['opt_ret_real'])
    ax.scatter(100 * ok['opt_ret'], 100 * ok['opt_ret_real'], s=14, label=a)
if crows:
    comp = pd.concat(crows, ignore_index=True)
    lim = np.nanmax(np.abs(100 * comp[['opt_ret', 'opt_ret_real']].to_numpy(float))) if len(comp) else 1
    ax.plot([-lim, lim], [-lim, lim], color='k', lw=0.7)
    ax.set_xlabel('model option return (% notional)'); ax.set_ylabel('real-quote option return (% notional)')
    ax.legend(fontsize=7, frameon=False); plt.tight_layout(); plt.show()
    ok = comp.dropna(subset=['opt_ret_real'])
    comp_summary = ok.groupby('asset').apply(lambda g: pd.Series({
        'n_pieces': len(g), 'corr': g['opt_ret'].corr(g['opt_ret_real']) if len(g) > 2 else np.nan,
        'model_sum_pct': 100 * g['opt_ret'].sum(), 'real_sum_pct': 100 * g['opt_ret_real'].sum(),
        'mean_diff_pct': 100 * (g['opt_ret_real'] - g['opt_ret']).mean(),
        'no_bid_exits': int((g['exit_flag'] == 'no_bid_intrinsic').sum())}), include_groups=False)
    display(comp_summary.round(3))
    print('entry quote missing:', comp['exit_flag'].eq('no_entry_quote').groupby(comp['asset']).sum().to_dict())
else:
    comp, comp_summary = pd.DataFrame(), pd.DataFrame()
    plt.close(fig); print('no option pieces inside the collector window')

## 7. Exports

In [ ]:
trades_out = []
for a, r in res.items():
    t = r['tr'].drop(columns=['i_entry', 'i_exit']).reset_index().rename(columns={'index': 'trade_id'})
    if r['ok_opt']:
        o = r['pcs'].groupby('trade_id').agg(opt_ret=('opt_ret', 'sum'), opt_pieces=('piece', 'size'),
                                             opt_q=('q', 'first'), opt_premium_share=('premium_share', 'first'))
        t = t.merge(o, left_on='trade_id', right_index=True, how='left')
    trades_out.append(t.assign(asset=a))
pd.concat(trades_out, ignore_index=True).to_csv(OUT / '01_trades.csv.gz', index=False, float_format='%.6g')
eq_out = pd.concat({a: d for a, d in daily.items()} | {'POOLED': pool}, names=['asset', 'date']).reset_index()
eq_out.to_csv(OUT / '01_equity_daily.csv', index=False, float_format='%.6g')
metrics.to_csv(OUT / '01_metrics.csv', index=False, float_format='%.6g')
decision_tab.to_csv(OUT / '01_null_decision.csv', index=False, float_format='%.6g')
pd.concat(nulls, names=['asset', 'i']).reset_index(level=0).to_csv(OUT / '01_null_runs.csv.gz', index=False, float_format='%.5g')
calib.to_csv(OUT / '01_option_calibration.csv', index=False, float_format='%.6g')
if len(comp):
    comp.to_csv(OUT / '01_option_real_vs_model.csv', index=False, float_format='%.6g')
    comp_summary.to_csv(OUT / '01_option_real_vs_model_summary.csv', float_format='%.6g')
cov.to_csv(OUT / '01_coverage.csv', index=False)
(OUT / '01_decision.json').write_text(json.dumps({'primary': primary, 'registration': bt.registration(P)}, indent=2, default=str))
(OUT / '01_run_manifest.json').write_text(json.dumps({
    'module': '01_breakout_backtest', 'version': bt.VERSION, 'synthetic': SYNTHETIC, 'assets': list(frames),
    'n_null': N_NULL, 'end': str(END), 'windows': {k: [str(x) for x in v] for k, v in WINDOWS.items()}}, indent=2))
print(sorted(p.name for p in OUT.iterdir()))